# Artificial Neural Network — GAT IOP Prediction

This notebook develops an Artificial Neural Network (ANN) to predict Applanation/GAT IOP from:

- iCare IOP
- Central Corneal Thickness (CCT)

The same patient-level train, validation, and test datasets used for the Random Forest are used here to allow a fair comparison between models.

The test dataset will remain untouched during model development and hyperparameter tuning.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [3]:
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.13.0+cu130
GPU available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
## Load Dataset

The ANN uses the same training, validation, and test splits as the Random Forest.

Features:
- iCare IOP
- CCT

Target:
- Applanation/GAT IOP

In [4]:
train = pd.read_csv("../data/splits/train.csv")
validation = pd.read_csv("../data/splits/validation.csv")
test = pd.read_csv("../data/splits/test.csv")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (4648, 5)
Validation: (1001, 5)
Test: (990, 5)


## Define Features and Target

The ANN receives two input features:

X = [iCare IOP, CCT]

The target is:

y = Applanation/GAT IOP

In [16]:
features = ["icare_iop", "cct"]
target = "applanation_iop"

X_train = train[features].values
y_train = train[target].values

X_validation = validation[features].values
y_validation = validation[target].values

X_test = test[features].values
y_test = test[target].values

## Feature Scaling

Neural networks generally train more effectively when input features are on similar numerical scales.

CCT is measured in micrometres (~480–580), while iCare IOP is measured in mmHg (~3–70).

We therefore standardize both features using:

z = (x − mean) / standard deviation

The scaler is fitted only on the training data. The same transformation is then applied to validation and test data to prevent data leakage.

In [19]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_validation_scaled = scaler.transform(X_validation)
X_test_scaled = scaler.transform(X_test)

## Convert Data to PyTorch Tensors

PyTorch neural networks operate on tensors rather than NumPy arrays.

We convert the scaled feature data and target values into PyTorch tensors.

In [20]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)

X_validation_tensor = torch.tensor(
    X_validation_scaled,
    dtype=torch.float32
)
y_validation_tensor = torch.tensor(
    y_validation,
    dtype=torch.float32
).reshape(-1, 1)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)
y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32
).reshape(-1, 1)

## Define the Neural Network

The first ANN uses a simple fully connected architecture:

2 inputs → 32 neurons → 16 neurons → 1 output

The two input neurons represent:

- iCare IOP
- CCT

The final output is the predicted Applanation/GAT IOP.

ReLU activation functions are used between the hidden layers to allow the network to learn nonlinear relationships.

In [22]:
class IOP_ANN(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = IOP_ANN().to(device)

print(model)
print("Device:", device)

## Loss Function and Optimizer

The neural network needs a way to measure how far its predictions are from the actual GAT IOP values.

We use Mean Squared Error (MSE) as the training loss.

The optimizer then adjusts the network's weights to reduce this loss.

We use the Adam optimizer for the initial model.

In [24]:
loss_function = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

## Prepare Training Batches

The training data is divided into smaller batches.

Instead of sending all 4,648 samples through the network at once, the ANN processes a fixed number of samples at a time.

After each batch, the model calculates its error and updates its weights.

In [25]:
batch_size = 32

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

In [26]:
for X_batch, y_batch in train_loader:

    X_batch = X_batch.to(device)
    y_batch = y_batch.to(device)

    optimizer.zero_grad()

    predictions = model(X_batch)

    loss = loss_function(predictions, y_batch)

    loss.backward()

    optimizer.step()

## Train the Neural Network

An epoch is one complete pass through the entire training dataset.

During each epoch, the model:

1. Processes the training data in batches.
2. Makes predictions.
3. Calculates the MSE loss.
4. Uses backpropagation to calculate gradients.
5. Uses Adam to update the weights.

We record the average training loss after each epoch so that we can monitor the learning process.

In [27]:
epochs = 200

train_losses = []

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        predictions = model(X_batch)

        # Calculate loss
        loss = loss_function(predictions, y_batch)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    train_losses.append(average_loss)

    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch {epoch + 1}/{epochs} "
            f"- Loss: {average_loss:.4f}"
        )

Epoch 20/200 - Loss: 14.7836
Epoch 40/200 - Loss: 14.6792
Epoch 60/200 - Loss: 14.8439
Epoch 80/200 - Loss: 14.7044
Epoch 100/200 - Loss: 14.7822
Epoch 120/200 - Loss: 14.6380
Epoch 140/200 - Loss: 14.6674
Epoch 160/200 - Loss: 14.5889
Epoch 180/200 - Loss: 14.6699
Epoch 200/200 - Loss: 14.6754


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(train_losses)

plt.xlabel("Epoch")
plt.ylabel("Training MSE Loss")
plt.title("ANN Training Loss")

plt.grid(True, alpha=0.3)
plt.show()

## Initial ANN Validation Evaluation

We now evaluate the trained ANN on the validation dataset using the same metrics used for the Random Forest.

The test dataset remains untouched.

In [29]:
model.eval()

with torch.no_grad():
    validation_predictions_tensor = model(
        X_validation_tensor.to(device)
    )

validation_predictions_ann = (
    validation_predictions_tensor
    .cpu()
    .numpy()
    .flatten()
)

ann_val_mae = mean_absolute_error(
    y_validation,
    validation_predictions_ann
)

ann_val_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        validation_predictions_ann
    )
)

ann_val_r2 = r2_score(
    y_validation,
    validation_predictions_ann
)

print("ANN Validation Results")
print("------------------")
print(f"MAE:  {ann_val_mae:.3f} mmHg")
print(f"RMSE: {ann_val_rmse:.3f} mmHg")
print(f"R²:   {ann_val_r2:.3f}")

ANN Validation Results
------------------
MAE:  2.847 mmHg
RMSE: 4.007 mmHg
R²:   0.675
